In [ ]:
# Install (jika perlu)
# !pip install nltk wordcloud

# Import
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
cv = pd.read_csv('/content/all_job_post.csv')
job = pd.read_json('/content/resumes_dataset.jsonl', lines=True)

# Display columns of 'job' to identify the correct text column
# print("Columns in job DataFrame:", job.columns.tolist())

# Once the correct column name is identified, update 'job['ResumeText']' accordingly.
# For now, this line will intentionally cause an error to prompt for the correct column.
df = pd.DataFrame({
    'cv_text': cv['job_description'],
    'job_desc': job['Text']
})

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/all_job_post.csv'

In [ ]:
#menyimpan sebagai main_data
df.to_csv('main_data.csv', index=False)

In [ ]:
# Cek missing value
print(df.isnull().sum())

# Hapus duplikasi
df = df.drop_duplicates()

# Hapus data kosong
df = df.dropna(subset=['cv_text', 'job_desc'])

df = df.reset_index(drop=True)

print("Shape:", df.shape)

In [ ]:
df.to_csv('cleaned_data.csv', index=False)

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('indonesian'))

def preprocess_text(text):
    text = str(text) # Convert to string to handle non-string types like floats
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

df['cv_clean'] = df['cv_text'].apply(preprocess_text)
df['job_clean'] = df['job_desc'].apply(preprocess_text)

df[['cv_clean', 'job_clean']].head()

In [ ]:
df.to_csv('processed_data.csv', index=False)

In [ ]:
#word cloud
text = " ".join(df['cv_clean'])

wc = WordCloud(width=800, height=400).generate(text)

plt.imshow(wc)
plt.axis('off')
plt.title("WordCloud CV")
plt.show()

In [ ]:
from collections import Counter

words = text.split()
top_words = Counter(words).most_common(10)

print("Top Words:", top_words)

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

cv_vectors = tfidf.fit_transform(df['cv_clean'])
jd_vectors = tfidf.transform(df['job_clean'])

print(f"Fitur TF-IDF ditingkatkan menjadi: {cv_vectors.shape[1]}")

In [ ]:
feature_names = tfidf.get_feature_names_out()

cv_tfidf_df = pd.DataFrame(cv_vectors.toarray(), columns=feature_names)
jd_tfidf_df = pd.DataFrame(jd_vectors.toarray(), columns=feature_names)

# Simpan ke CSV
cv_tfidf_df.to_csv('cv_tfidf.csv', index=False)
jd_tfidf_df.to_csv('jd_tfidf.csv', index=False)

# Simpan Model & Vector menggunakan joblib
joblib.dump(cv_vectors, 'cv_vectors.pkl')
joblib.dump(jd_vectors, 'jd_vectors.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

In [ ]:
# from google.colab import files

# #download seluruh file
# files.download('main_data.csv')
# files.download('cleaned_data.csv')
# files.download('processed_data.csv')
# files.download('cv_tfidf.csv')
# files.download('jd_tfidf.csv')
# files.download('cv_vectors.pkl')
# files.download('jd_vectors.pkl')
# files.download('tfidf_vectorizer.pkl')

### AI Engineer

In [ ]:
!pip install tensorflowjs

In [ ]:
import random
import tensorflow as tf

def set_reproducibility(seed=42):
    """Menetapkan seed untuk memastikan hasil yang konsisten (Reproducible)"""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.config.experimental.enable_op_determinism()
    print(f"\u2705 Random seed set to {seed}. Hasil training akan lebih konsisten.")

set_reproducibility(42)

### Data Preparation and Splitting

In [ ]:
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(cv['category'])
num_classes = len(le.classes_)

X_train, X_val, y_train, y_val = train_test_split(
    cv_vectors.toarray(), y, test_size=0.2, random_state=42
)

print(f"Input shape: {X_train.shape[1]}")
print(f"Total Classes: {num_classes}")

### Model Architecture Definition



In [ ]:
class CustomTrainingCallback(tf.keras.callbacks.Callback):
    """Custom Callback untuk memantau target akurasi 85%"""
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('accuracy') > 0.85:
            print(f"\nReached 85% accuracy at epoch {epoch}, stopping training!")
            self.model.stop_training = True

def build_model(input_shape, num_classes):
    inputs = layers.Input(shape=(input_shape,))
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_model(X_train.shape[1], num_classes)
model.summary()

### Model Training

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[CustomTrainingCallback()],
    verbose=1
)

### Model Saving

In [ ]:
model_path = 'tech_hire_model.keras'
model.save(model_path)

print(f"Model berhasil disimpan di: {model_path}")

from google.colab import files
files.download(model_path)

### Model Inference

In [ ]:
loaded_model = tf.keras.models.load_model('tech_hire_model.keras')
vectorizer = joblib.load('tfidf_vectorizer.pkl')

def predict_cv_category(raw_text):
    cleaned_text = preprocess_text(raw_text)

    tfidf_matrix = vectorizer.transform([cleaned_text])
    input_vector = tfidf_matrix.toarray()

    prediction = loaded_model.predict(input_vector, verbose=0)
    class_idx = np.argmax(prediction)
    confidence = np.max(prediction)

    # Menggunakan le.classes_ yang sudah didefinisikan sebelumnya
    category_name = le.classes_[class_idx]

    return category_name, confidence

sample_cv = "Senior Software Engineer with expertise in Python, Cloud Infrastructure, and Microservices architecture."
category, score = predict_cv_category(sample_cv)

print(f"Teks CV: {sample_cv}")
print(f"Hasil Prediksi: {category}")
print(f"Confidence Score: {score:.4f}")

### Retraining Model

In [ ]:
def build_v9_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

X_train, X_val, y_train, y_val = train_test_split(
    cv_vectors.toarray(), y, test_size=0.2, random_state=42
)

input_dim = X_train.shape[1]
model_v9 = build_v9_model(input_dim, num_classes)

model_v9.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',    metrics=['accuracy']
)

print("Retraining Model v9 dengan 5000 fitur...")
history_v9 = model_v9.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    verbose=1
)

production_model = model_v9
production_vectorizer = tfidf

In [ ]:
def predict_v9(text):
    clean = preprocess_text(text)
    vec = tfidf.transform([clean]).toarray()
    pred = model_v9.predict(vec, verbose=0)
    idx = np.argmax(pred)
    return le.classes_[idx], np.max(pred)

test_samples = [
    ('SALES', 'Sales Executive with a track record of exceeding quotas. Skilled in cold calling and lead generation.'),
    ('BUSINESS-DEVELOPMENT', 'Strategic partnerships manager focused on market expansion and revenue growth.'),
    ('FINANCE', 'Chartered Accountant with 10 years experience in auditing and financial reporting.'),
    ('HR', 'HR Specialist with expertise in talent acquisition and employee relations.'),
    ('INFORMATION-TECHNOLOGY', 'Senior Software Engineer specializing in Python and Cloud Infrastructure.')
]

print(f"{'Expected Label':<25} | {'Predicted (v9)':<25} | {'Status'}")
print("-" * 75)

for expected, text in test_samples:
    p_cat, p_conf = predict_v9(text)
    status = '\u2705' if p_cat == expected else '\u274c'
    print(f"{expected:<25} | {p_cat:<25} | {status} ({p_conf*100:.1f}%)")

### Model Evaluation dengan Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

y_pred_v9_probs = model_v9.predict(X_val, verbose=0)
y_pred_v9_classes = np.argmax(y_pred_v9_probs, axis=1)

print("--- Model v9 Detailed Classification Report ---")
print(classification_report(y_val, y_pred_v9_classes, target_names=le.classes_))

cm_v9 = confusion_matrix(y_val, y_pred_v9_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_v9, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Model v9 (Deep Architecture)')
plt.show()

In [ ]:
model_v9.save('tech_hire_model.keras')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

production_model = tf.keras.models.load_model('tech_hire_model.keras')
production_vectorizer = joblib.load('tfidf_vectorizer.pkl')

def predict_production_final(text):
    clean = preprocess_text(text)
    vec = production_vectorizer.transform([clean]).toarray()
    pred = production_model.predict(vec, verbose=0)
    idx = np.argmax(pred)
    conf = np.max(pred)
    return le.classes_[idx], conf

print("✓ Model & Vectorizer berhasil disimpan dengan nama standar.")

In [ ]:
print(f"{'Expected Label':<25} | {'v9 Prediction':<25} | {'Status'}")
print("-" * 75)

for expected, text in test_samples:
    p_cat, p_conf = predict_production_final(text)
    status = "✅" if p_cat == expected else "❌"
    print(f"{expected:<25} | {p_cat:<25} | {status} ({p_conf*100:.1f}%)")

### Data Augmentation

In [ ]:
def simple_augment(text):
    words = text.split()
    if len(words) < 5: return text
    n = max(1, int(len(words) * 0.1))
    for _ in range(n):
        idx = random.randint(0, len(words) - 1)
        words.pop(idx)
    return " ".join(words)

target_categories = ['SALES', 'BUSINESS-DEVELOPMENT']
df_target = df[df['cv_text'].str.upper().isin(target_categories)].copy()

augmented_data = []
for _, row in df_target.iterrows():
    augmented_data.append({
        'cv_text': simple_augment(row['cv_text']),
        'category': row['cv_text'].upper() if 'category' not in df.columns else row['category'] # Fallback logic
    })

df_aug = pd.DataFrame(augmented_data)
print(f"Menambahkan {len(df_aug)} sampel hasil augmentasi.")

if 'category' not in df.columns:
    df['category'] = cv['category'].values

df_final = pd.concat([df, df_aug], ignore_index=True)
print(f"Total data setelah augmentasi: {len(df_final)}")

### Retraining Model

In [ ]:
tfidf_aug = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_final_tfidf = tfidf_aug.fit_transform(df_final['cv_clean'] if 'cv_clean' in df_final.columns else df_final['cv_text'].apply(preprocess_text))

le_final = LabelEncoder()
y_final = le_final.fit_transform(df_final['category'])

X_train_f, X_val_f, y_train_f, y_val_f = train_test_split(
    X_final_tfidf.toarray(), y_final, test_size=0.2, random_state=42
)

model_v10 = build_model(X_train_f.shape[1], len(le_final.classes_))
history_v10 = model_v10.fit(
    X_train_f, y_train_f,
    validation_data=(X_val_f, y_val_f),
    epochs=30,
    batch_size=32,
    verbose=1
)

production_model = model_v10
production_vectorizer = tfidf_aug

### Model Evaluation (Augmented Data)

In [ ]:
def predict_v10(text):
    clean = preprocess_text(text)
    vec = production_vectorizer.transform([clean]).toarray()
    pred = production_model.predict(vec, verbose=0)
    idx = np.argmax(pred)
    return le_final.classes_[idx], np.max(pred)

print(f"{'Expected Label':<25} | {'Predicted (v10)':<25} | {'Status'}")
print("-" * 75)

for expected, text in test_samples:
    p_cat, p_conf = predict_v10(text)
    status = '✅' if p_cat == expected else '❌'
    print(f"{expected:<25} | {p_cat:<25} | {status} ({p_conf*100:.1f}%)")

In [ ]:
y_pred_v10_probs = production_model.predict(X_val_f, verbose=0)
y_pred_v10_classes = np.argmax(y_pred_v10_probs, axis=1)

cm_v10 = confusion_matrix(y_val_f, y_pred_v10_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_v10, annot=True, fmt='d', xticklabels=le_final.classes_, yticklabels=le_final.classes_, cmap='Greens')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Model v10 (After Augmentation)')
plt.show()

### Advanced Data Augmentation

In [ ]:
def advanced_augment(text):
    words = text.split()
    if len(words) < 10: return text
    n = int(len(words) * 0.15)
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(words)), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]
    return " ".join(words)

target_cats = ['SALES', 'BUSINESS-DEVELOPMENT']
df_to_augment = df[df['category'].isin(target_cats)].copy()

augmented_list = []
for _, row in df_to_augment.iterrows():
    for _ in range(2):
        augmented_list.append({
            'cv_text': row['cv_text'],
            'category': row['category'],
            'cv_clean': advanced_augment(row['cv_clean'])
        })

df_aug_new = pd.DataFrame(augmented_list)
print(f"Sampel baru yang dihasilkan: {len(df_aug_new)}")

df_v10_final = pd.concat([df, df_aug_new], ignore_index=True)
print(f"Total dataset untuk v10 Fine-Tuning: {len(df_v10_final)}")

### Model Fine-Tuning

In [ ]:
tfidf_v10 = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_v10 = tfidf_v10.fit_transform(df_v10_final['cv_clean']).toarray()

le_v10 = LabelEncoder()
y_v10 = le_v10.fit_transform(df_v10_final['category'])

X_train_v10, X_val_v10, y_train_v10, y_val_v10 = train_test_split(
    X_v10, y_v10, test_size=0.2, random_state=42, stratify=y_v10
)

model_v10_ft = build_model(X_train_v10.shape[1], len(le_v10.classes_))

history_v10_ft = model_v10_ft.fit(
    X_train_v10, y_train_v10,
    validation_data=(X_val_v10, y_val_v10),
    epochs=30,
    batch_size=32,
    verbose=1
)

production_model = model_v10_ft
production_vectorizer = tfidf_v10
le_final = le_v10

### Fine-Tuned Model Evaluation

In [ ]:
y_pred_ft = np.argmax(model_v10_ft.predict(X_val_v10, verbose=0), axis=1)
print("\n--- Classification Report Model v10 (Fine-Tuned) ---")
print(classification_report(y_val_v10, y_pred_ft, target_names=le_v10.classes_))

In [ ]:
cm_v10_ft = confusion_matrix(y_val_v10, y_pred_ft)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_v10_ft,
    annot=True,
    fmt='d',
    xticklabels=le_v10.classes_,
    yticklabels=le_v10.classes_,
    cmap='Purples'
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Model v10 Fine-Tuned (Augmented Data)')
plt.show()

sales_idx = list(le_v10.classes_).index('SALES')
biz_dev_idx = list(le_v10.classes_).index('BUSINESS-DEVELOPMENT')

print(f"Correctly predicted SALES: {cm_v10_ft[sales_idx, sales_idx]}")
print(f"SALES misclassified as BUSINESS-DEVELOPMENT: {cm_v10_ft[sales_idx, biz_dev_idx]}")
print(f"Correctly predicted BUSINESS-DEVELOPMENT: {cm_v10_ft[biz_dev_idx, biz_dev_idx]}")
print(f"BUSINESS-DEVELOPMENT misclassified as SALES: {cm_v10_ft[biz_dev_idx, sales_idx]}")

### TensorBoard Setup Untuk Monitoring

In [ ]:
import shutil
import datetime
import os

if os.path.exists('logs'):
    shutil.rmtree('logs')
    print("✅ Log lama telah dihapus.")

log_dir = "logs/fit/v10_finetuned_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(f"⌑ Log TensorBoard akan disimpan di: {log_dir}")

In [ ]:
class EarlyStopAt99(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get('val_accuracy')
        if val_acc is not None and val_acc >= 0.99:
            print(f"\nReached 99% validation accuracy at epoch {epoch}, stopping training!")
            self.model.stop_training = True

model_v10_final = build_model(X_train_v10.shape[1], len(le_v10.classes_))
stop_at_99 = EarlyStopAt99()

history_final = model_v10_final.fit(
    X_train_v10, y_train_v10,
    validation_data=(X_val_v10, y_val_v10),
    epochs=30,
    batch_size=32,
    verbose=1,
    callbacks=[tensorboard_callback, stop_at_99]
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

### Saving Final Model & Assets

In [ ]:
# Menyimpan Model Produksi Akhir (Final)
model_v10_ft.save('tech_hire_model_final.keras')
joblib.dump(tfidf_v10, 'tfidf_vectorizer_final.pkl')
joblib.dump(le_v10, 'label_encoder_final.pkl')

print("✓ Model Final: tech_hire_model_final.keras")
print("✓ Vectorizer Final: tfidf_vectorizer_final.pkl")
print("✓ Label Encoder Final: label_encoder_final.pkl")

In [ ]:
# from google.colab import files

# files.download('tech_hire_model_final.keras')
# files.download('tfidf_vectorizer_final.pkl')
# files.download('label_encoder_final.pkl')

### Menguji Model Produksi Akhir (Inference)

In [ ]:
# Memuat model produksi akhir
loaded_model = tf.keras.models.load_model('tech_hire_model_final.keras')
loaded_tfidf = joblib.load('tfidf_vectorizer_final.pkl')
loaded_le = joblib.load('label_encoder_final.pkl')

def predict_cv(text):
    clean_text = preprocess_text(text)
    vec = loaded_tfidf.transform([clean_text]).toarray()
    preds = loaded_model.predict(vec, verbose=0)
    class_idx = np.argmax(preds)
    confidence = np.max(preds)
    return loaded_le.classes_[class_idx], confidence

# Menggunakan sampel yang lebih deskriptif agar confidence tinggi
final_test_samples = [
    ("Experienced Human Resources Director specializing in employee relations, talent acquisition strategies, workforce planning, and corporate policy development."),
    ("Senior Financial Analyst with extensive background in budget forecasting, fiscal reporting, investment auditing, and complex financial modeling."),
    ("Senior Full Stack Software Engineer proficient in Python, Java, and Cloud Infrastructure using AWS, Docker, and Kubernetes for scalable systems."),
    ("Top-performing Sales Executive with a track record of exceeding revenue targets, managing B2B client portfolios, and executing lead generation campaigns."),
    ("Strategic Business Development Manager focused on market expansion, partnership negotiations, and identifying new revenue streams through strategic alliances.")
]

print(f"{'Predicted Category':<25} | {'Confidence'}")
print("-" * 45)

for sample_text in final_test_samples:
    category, score = predict_cv(sample_text)
    print(f"{category:<25} | {score*100:>9.2f}%")
    print(f"Teks: {sample_text[:80]}...")
    print("-" * 45)